#Bangla Dual-Channel Conversational Dataset Builder (FINAL)

**Pipeline:** Mono conversation → Bangla Diarization → Speech Separation (MossFormer2) → Dual-Channel (Stereo) Output

This notebook converts mono Bangla conversation audio into a dual-channel (stereo) dataset suitable for training full-duplex speaking models. It follows the metadata-first strategy (DuplexChat-style) so you can build an **open-source, distributable manifest** without redistributing copyrighted audio.

### Stages
1. Environment Setup & Hugging Face Auth
2. Ingest & Normalize Audio
3. Bangla Speaker Diarization
4. Speech Separation (MossFormer2 via ClearerVoice fork)
5. Dual-Channel Assembly (frame-level routing)
6. Quality Filtering & Manifest Generation

### Prerequisites
- **Runtime:** Colab T4 GPU (`Runtime → Change runtime type → T4 GPU`)
- **HF Token:** Add `HF_TOKEN` in Colab Secrets (🔑 icon in left sidebar) with a *Read* token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- **Accept model terms (one-time):** On Hugging Face, click *Agree and access repository* on **both**
  [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) and
  [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0).
- **Audio:** Upload a **short** Bangla conversation clip (30–60 seconds). For long podcasts, split into segments first.

### 🔑 What makes this version work
| Fix | Cell |
|---|---|
| `pyannote.audio==4.0.7` pinned (3.x is a dead zone with `huggingface_hub` 1.x) | 1.1 |
| `numba>=0.67` for NumPy 2.5 support | 1.1 |
| `yamlargparse` explicitly installed (clearvoice `--no-deps` skips it) | 1.1 |
| `torchaudio.AudioMetaData` monkey-patch before pyannote imports | 1.1 |
| ClearerVoice **fork** (numpy-2 compatible) + `--no-deps` | 1.1 |
| `token=` kwarg (pyannote 4.x syntax) | 3.1 |
| `DiarizeOutput` vs `Annotation` compatibility | 3.2 |
| Recursive stream extraction from ClearVoice output | 4.2 |
| Stream sanity check (silence / identical / length) | 4.3 |
| **Frame-level routing with overlap-aware freeze** | **5.1** |
| Self-contained purity score (no cross-cell deps) | 6.1 |
| Single zip with orig + speaker_1 + speaker_2 | 6.3 |

## ⚙️ Stage 1: Environment Setup & Authentication

In [ ]:
# Cell 1.1 — Environment setup (REVISED: robust MossFormer2 check) [PERMANENT]
import subprocess, os, sys, importlib

def sh(cmd, quiet=False):
    if not quiet: print(f"→ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        if "already satisfied" not in err.lower() and "dependency resolver" not in err.lower():
            print(f"  ⚠️  {err[-400:]}")
    return r.returncode == 0

print("=" * 65)
print(f"Python: {sys.version.split()[0]}  |  Platform: {sys.platform}")
print("=" * 65, "\n")

# --- 1. numba FIRST (its numpy ceiling breaks the import chain) ---
print("[1/7] numba (must be >=0.67 for numpy 2.5)")
sh('pip install -q --upgrade "numba>=0.67.0,<0.68"')

# --- 2. pyannote.audio 4.x (3.x is dead zone with hf_hub 1.x) ---
print("\n[2/7] pyannote.audio==4.0.7")
sh('pip install -q --upgrade --force-reinstall "pyannote.audio==4.0.7"')

# --- 3. Audio libs ---
print("\n[3/7] audio libs")
sh('pip install -q --upgrade "librosa>=0.10.2" "soundfile>=0.12" "scipy>=1.10" yamlargparse')

# --- 4. MONKEY-PATCH torchaudio BEFORE pyannote imports it ---
print("\n[4/7] torchaudio compatibility patch")
import torchaudio as _ta
if not hasattr(_ta, "AudioMetaData"):
    class _AudioMetaData:
        sample_rate: int = 0
        num_frames: int = 0
        num_channels: int = 0
        bits_per_sample: int = 0
        encoding: str = ""
    _ta.AudioMetaData = _AudioMetaData
    print("  ✓ patched torchaudio.AudioMetaData")
else:
    print("  ✓ torchaudio.AudioMetaData already present")
if not hasattr(_ta, "list_audio_backends"):
    _ta.list_audio_backends = lambda: ["sox_io"]
    print("  ✓ patched torchaudio.list_audio_backends")
else:
    print("  ✓ torchaudio.list_audio_backends already present")

# --- 5. ClearerVoice fork clone ---
print("\n[5/7] ClearerVoice fork")
FORK_DIR  = "ClearerVoice-Studio-fork"
FORK_REPO = "https://github.com/meizhong986/ClearerVoice-Studio.git"
if not os.path.isdir(FORK_DIR):
    if not sh(f'git clone -q {FORK_REPO} {FORK_DIR}', quiet=True):
        print("  ❌ Clone failed")
        raise SystemExit(1)
    print(f"  Cloned → ./{FORK_DIR}")
else:
    print(f"  Already present → ./{FORK_DIR}")

# --- 6. Install clearvoice WITH dependencies ---
print("\n[6/7] clearvoice (full dependency resolution)")
sh('pip uninstall -y -q clearvoice', quiet=True)

sh(
    'pip install -q --upgrade '
    'python_speech_features opencv-python rotary_embedding_torch '
    'torchinfo einops pydub attrs absl-py'
)

if not sh(f'pip install -q -e ./{FORK_DIR}/clearvoice'):
    print("  ⚠️  Editable install failed — upgrading setuptools and retrying")
    sh('pip install -q --upgrade setuptools wheel')
    if not sh(f'pip install -q -e ./{FORK_DIR}/clearvoice'):
        print("  ❌ clearvoice install failed")
        raise SystemExit(1)

# --- 7. VERIFICATION ---
print("\n" + "=" * 65)
print("VERIFICATION")
print("=" * 65)

import importlib.metadata as md
fatal, warn = [], []

def check(name, fn, version_fn=None):
    try:
        mod = fn()
        ver = version_fn() if version_fn else getattr(mod, "__version__", "?")
        print(f"✅ {name:15s} {ver}")
    except Exception as e:
        print(f"❌ {name:15s} {type(e).__name__}: {str(e)[:120]}")
        fatal.append(name)

check("numpy",       lambda: __import__("numpy"))
check("numba",       lambda: __import__("numba"))
check("librosa",     lambda: __import__("librosa"))
check("soundfile",   lambda: __import__("soundfile"))
check("scipy",       lambda: __import__("scipy"))
check("yamlargparse", lambda: __import__("yamlargparse"))
check("pyannote",    lambda: __import__("pyannote.audio"),
      lambda: md.version("pyannote.audio"))
check("clearvoice",  lambda: __import__("clearvoice"))

# The full chain that matters
try:
    from pyannote.audio import Pipeline  # noqa
    from clearvoice import ClearVoice     # noqa
    print("✅ pyannote Pipeline + clearvoice ClearVoice import cleanly")
except Exception as e:
    print(f"❌ pipeline import chain  {type(e).__name__}: {e}")
    fatal.append("pipeline imports")

# --- MossFormer2: discovery-based check (no fragile class-name import) ---
# ClearerVoice resolves 'MossFormer2_SS_16K' via a string registry in Cell 4.1,
# so the class does NOT need to be directly importable here. Verify that the
# module tree loads and that a MossFormer2-named symbol exists *somewhere*.
try:
    import pkgutil
    _m2ss = importlib.import_module("clearvoice.models.mossformer2_ss")
    _found = []
    for _n in dir(_m2ss):
        if "mossformer" in _n.lower() and not _n.startswith("_"):
            _found.append(_n)
    if not _found:
        # Look one level deeper — class may live in a submodule not re-exported
        for _imp, _name, _ispkg in pkgutil.iter_modules(
                _m2ss.__path__, _m2ss.__name__ + "."):
            try:
                _sub = importlib.import_module(_name)
                for _n in dir(_sub):
                    if "mossformer" in _n.lower() and not _n.startswith("_"):
                        _found.append(f"{_name}.{_n}")
            except Exception:
                pass
    if _found:
        print(f"✅ MossFormer2 available          {_found[:3]}")
    else:
        print("✅ MossFormer2 module loads        "
              "(class resolved via ClearVoice registry at Cell 4.1)")
except Exception as e:
    print(f"❌ MossFormer2 module import  {type(e).__name__}: {e}")
    fatal.append("MossFormer2 import")

try:
    import torch
    if torch.cuda.is_available():
        print(f"✅ CUDA           {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  CUDA           not available")
        warn.append("No GPU — set Runtime → Change runtime type → T4 GPU")
except Exception as e:
    print(f"❌ torch          {e}")
    fatal.append("torch")

print("=" * 65)
if fatal:
    print("❌ ENVIRONMENT NOT READY")
    print(f"   Failures: {fatal}\n")
    print("   Recovery:")
    print("   1. Runtime → Restart runtime")
    print("   2. Re-run this cell from the top")
    print("   3. If still failing, copy THIS ENTIRE output and share it")
    raise SystemExit(1)

for w in warn: print(f"⚠️  {w}")
print("✅ ENVIRONMENT READY — proceed to Cell 1.2")
print("=" * 65)

In [ ]:
# Cell 1.2 — Authenticate with Hugging Face
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Hugging Face login successful.")
except Exception as e:
    print(f"❌ Login failed: {e}")
    print("Add HF_TOKEN in Colab Secrets (🔑 icon) with a Read token.")

## 📥 Stage 2: Ingest & Normalize Audio

In [ ]:
# Cell 2.1 — Upload your audio file
from google.colab import files
import os

print("Upload a short Bangla conversation clip (30–60s recommended):")
uploaded = files.upload()
AUDIO_PATH = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {AUDIO_PATH} ({os.path.getsize(AUDIO_PATH)/1024:.1f} KB)")

In [ ]:
# Cell 2.2 — Load, convert to mono, resample to 16 kHz
import torchaudio
import torch

TARGET_SR = 16000    # internal processing rate — locked to 16 kHz (diarization + MossFormer2_SS_16K)
OUTPUT_SR = 16000    # output WAV rate: set to 16000 or 24000 (or 8000/48000 if you prefer)

assert OUTPUT_SR in (16000, 24000), f"Unsupported OUTPUT_SR={OUTPUT_SR}; use 16000 or 24000"

def load_and_normalize(audio_path, target_sr=TARGET_SR):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
    return waveform, target_sr

waveform, sample_rate = load_and_normalize(AUDIO_PATH)
duration = waveform.shape[1] / sample_rate
print(f"✅ Loaded: {waveform.shape} | internal SR: {sample_rate} Hz | output SR: {OUTPUT_SR} Hz")
print(f"   Duration: {duration:.2f}s")

## 🎙️ Stage 3: Bangla Speaker Diarization

Uses `pyannote/speaker-diarization-3.1` with `pyannote.audio==4.0.7` (pinned in Cell 1.1).
The model is a HF Hub artifact, independent of the package version — pyannote 4.x loads it fine.

**Alternative:** `pyannote/speaker-diarization-community-1` is the newer 4.x-native pipeline. If the 3.1 model misbehaves, swap it in Cell 3.1.

In [49]:
# Cell 3.1 — Load the diarization model
from pyannote.audio import Pipeline
import torch

DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"
# If 3.1 fails, try: "pyannote/speaker-diarization-community-1"

diarization_pipeline = Pipeline.from_pretrained(
    DIARIZATION_MODEL,
    token=hf_token,
)
diarization_pipeline.to(torch.device("cuda"))
print(f"✅ Loaded diarization model: {DIARIZATION_MODEL}")

✅ Loaded diarization model: pyannote/speaker-diarization-3.1


In [ ]:
# Cell 3.2 — Run diarization (force 2 speakers)
output = diarization_pipeline(
    {"waveform": waveform, "sample_rate": sample_rate},
    min_speakers=2, max_speakers=2,
)

# pyannote.audio >=3.3 returns DiarizeOutput; older returns Annotation.
annotation = getattr(output, "speaker_diarization", output)

speaker_turns = [
    {"start": turn.start, "end": turn.end, "speaker": speaker}
    for turn, _, speaker in annotation.itertracks(yield_label=True)
]
speaker_turns.sort(key=lambda x: x["start"])

labels = sorted({t["speaker"] for t in speaker_turns})
print(f"✅ Found {len(speaker_turns)} turns across speakers: {labels}\n")
for t in speaker_turns[:10]:
    print(f"  [{t['start']:6.2f}s → {t['end']:6.2f}s]  {t['speaker']}")

In [51]:
# Cell 3.3 — Free diarization VRAM before separation (prevents T4 OOM)
import gc
diarization_pipeline.to(torch.device("cpu"))
gc.collect()
torch.cuda.empty_cache()
print("✅ Diarization pipeline moved to CPU to free VRAM for stage 4.")

✅ Diarization pipeline moved to CPU to free VRAM for stage 4.


## 🔊 Stage 4: Speech Separation (MossFormer2)

Unmixes the mono waveform into two separate speaker streams using ClearerVoice's MossFormer2 fork.

In [52]:
# Cell 4.1 — Initialize MossFormer2 separator
from clearvoice import ClearVoice

separator = ClearVoice(task='speech_separation', model_names=['MossFormer2_SS_16K'])
print("✅ MossFormer2 separator initialized.")

✅ MossFormer2 separator initialized.


In [ ]:
# Cell 4.2 — Run separation (handles all ClearVoice output shapes)
import numpy as np
import torchaudio

TEMP_MIX = "/content/temp_mix.wav"
torchaudio.save(TEMP_MIX, waveform, sample_rate)

output = separator(input_path=TEMP_MIX, online_write=False)

# Diagnostic dump
def describe(x, depth=0, name="output"):
    pad = "  " * depth
    t = type(x).__name__
    if isinstance(x, dict):
        print(f"{pad}{name}: dict, keys={list(x.keys())}")
        for k, v in x.items():
            describe(v, depth + 1, f"[{k!r}]")
    elif isinstance(x, (list, tuple)):
        print(f"{pad}{name}: {t}, len={len(x)}")
        if x: describe(x[0], depth + 1, f"{name}[0]")
    elif isinstance(x, np.ndarray):
        print(f"{pad}{name}: ndarray, shape={x.shape}, dtype={x.dtype}")
    elif hasattr(x, "shape") and hasattr(x, "dtype"):
        print(f"{pad}{name}: {t}, shape={tuple(x.shape)}, dtype={x.dtype}")
    else:
        print(f"{pad}{name}: {t}")

print("---- output structure ----")
describe(output)
print("--------------------------")

def to_1d(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float32).flatten()

def collect_streams(node):
    found = []
    if node is None: return found
    if isinstance(node, dict):
        for v in node.values(): found += collect_streams(v)
    elif isinstance(node, (list, tuple)):
        for v in node: found += collect_streams(v)
    elif isinstance(node, np.ndarray) or hasattr(node, "detach"):
        arr = to_1d(node)
        if arr.size > 1000: found.append(arr)
    return found

streams = collect_streams(output)
if len(streams) < 2:
    raise RuntimeError(
        f"Found only {len(streams)} audio stream(s). "
        f"Inspect the structure dump above.")

streams = sorted(streams, key=len, reverse=True)[:2]
stream_1, stream_2 = streams[0], streams[1]

print("✅ Separation complete.")
print(f"   Stream 1 length: {len(stream_1)}")
print(f"   Stream 2 length: {len(stream_2)}")

In [ ]:
# Cell 4.3 — Sanity-check separated streams
import numpy as np

problems = []
for i, s in enumerate([stream_1, stream_2], 1):
    if not np.isfinite(s).all():
        problems.append(f"stream_{i} contains NaN/Inf")
    rms = float(np.sqrt(np.mean(s ** 2)))
    if rms < 1e-5:
        problems.append(f"stream_{i} is silent (rms={rms:.2e})")

if len(stream_1) == len(stream_2) and np.allclose(stream_1, stream_2, atol=1e-6):
    problems.append("stream_1 == stream_2 (extractor picked same tensor twice)")

expected = int(duration * sample_rate)
for i, s in enumerate([stream_1, stream_2], 1):
    ratio = len(s) / expected
    if not (0.8 <= ratio <= 1.2):
        problems.append(f"stream_{i} length {len(s)} is {ratio:.2f}× expected")

if problems:
    print("❌ STREAM SANITY FAILED:")
    for p in problems: print(f"   • {p}")
    raise SystemExit(1)
print("✅ Streams passed sanity check — proceed to Cell 5.1.")

## 🎛️ Stage 5: Dual-Channel Assembly

Frame-level routing with overlap-aware freeze. Catches MossFormer2's mid-clip output swaps
(which happen during simultaneous speech) and holds identity through overlap regions.

In [ ]:
# Cell 5.1 — Frame-level routing with overlap-aware freeze [PERMANENT / FINAL]
import numpy as np
from scipy.signal import medfilt

FRAME_S  = 0.2   # 200ms frames
HOP_S    = 0.1   # 100ms hop
DB_CLEAR = 6.0   # >=6 dB dominance = confident vote
MEDIAN_S = 1.0   # 1s median filter
FADE_MS  = 100   # 100ms crossfade at transitions

if len(labels) < 2:
    raise RuntimeError(f"Expected 2 speakers, got {len(labels)}: {labels}")

speaker_A, speaker_B = labels[0], labels[1]
n  = min(len(stream_1), len(stream_2))
s1 = np.asarray(stream_1[:n], dtype=np.float32)
s2 = np.asarray(stream_2[:n], dtype=np.float32)

frame_len = int(FRAME_S * sample_rate)
hop_len   = int(HOP_S   * sample_rate)
n_frames  = max(1, (n - frame_len) // hop_len + 1)

e1 = np.array([np.mean(s1[i*hop_len : i*hop_len+frame_len].astype(np.float64)**2)
               for i in range(n_frames)])
e2 = np.array([np.mean(s2[i*hop_len : i*hop_len+frame_len].astype(np.float64)**2)
               for i in range(n_frames)])
db_diff = 10.0 * np.log10((e1 + 1e-12) / (e2 + 1e-12))

frame_t = (np.arange(n_frames) * hop_len + frame_len / 2) / sample_rate
a_act = np.zeros(n_frames, dtype=bool)
b_act = np.zeros(n_frames, dtype=bool)
for t in speaker_turns:
    m = (frame_t >= t["start"]) & (frame_t < t["end"])
    if t["speaker"] == speaker_A: a_act |= m
    else:                         b_act |= m

votes = np.full(n_frames, -1, dtype=np.int8)
for i in range(n_frames):
    if a_act[i] and not b_act[i]:
        if   db_diff[i] >  DB_CLEAR: votes[i] = 0
        elif db_diff[i] < -DB_CLEAR: votes[i] = 1
    elif b_act[i] and not a_act[i]:
        if   db_diff[i] < -DB_CLEAR: votes[i] = 0
        elif db_diff[i] >  DB_CLEAR: votes[i] = 1

filled = np.full(n_frames, -1, dtype=np.int8)
last = None
for i in range(n_frames):
    if votes[i] != -1:
        last = votes[i]
    if last is not None:
        filled[i] = last

first_valid = next((i for i in range(n_frames) if filled[i] != -1), None)
if first_valid is None:
    raise RuntimeError(f"No confident frames (DB_CLEAR={DB_CLEAR} dB). "
                       "Try DB_CLEAR=3.0 or a cleaner clip.")
filled[:first_valid] = filled[first_valid]

k = max(3, int(MEDIAN_S / HOP_S) | 1)
filled = medfilt(filled, kernel_size=k).astype(np.int8)

swap = np.zeros(n, dtype=np.float32)
for i in range(n_frames):
    end = min(n, i*hop_len + hop_len)
    swap[i*hop_len:end] = 0.0 if filled[i] == 0 else 1.0
if n > n_frames * hop_len:
    swap[n_frames*hop_len:] = swap[n_frames*hop_len - 1]

half = int(FADE_MS * sample_rate / 1000 / 2)
transitions = np.where(np.diff(filled) != 0)[0] + 1
for t_idx in transitions:
    sc = t_idx * hop_len
    a, b = max(0, sc - half), min(n, sc + half)
    if b > a:
        swap[a:b] = np.linspace(swap[a], swap[b-1], b - a, dtype=np.float32)

left_channel  = s1 * (1.0 - swap) + s2 * swap
right_channel = s2 * (1.0 - swap) + s1 * swap

n_voted = int((votes != -1).sum())
print(f"Frames: {n_frames} @ {FRAME_S*1000:.0f}ms | voted: {n_voted} | held: {n_frames-n_voted}")
if len(transitions) == 0:
    print("  ⚠️  No routing swaps detected (leakage-only case).")
else:
    print(f"  ⚠️  {len(transitions)} routing swap(s) applied:")
    for t_idx in transitions:
        ts = t_idx * hop_len / sample_rate
        hb = "stream_1" if filled[t_idx-1] == 0 else "stream_2"
        ha = "stream_1" if filled[t_idx]   == 0 else "stream_2"
        print(f"     [{ts:6.2f}s]  {hb} → {ha} holds {speaker_A}")

print(f"\n✅ Left = {speaker_A}, Right = {speaker_B}")

## ✅ Stage 6: Quality Filtering & Manifest Generation

In [56]:
# Cell 6.1 — Channel purity score (self-contained, no cross-cell deps)
import numpy as np

_missing = [v for v in ("left_channel", "right_channel", "speaker_turns",
                        "labels", "sample_rate") if v not in globals()]
if _missing:
    raise RuntimeError(f"Missing variables: {_missing}. Run Cell 5.1 first.")

def _energy_in_turn(stream, turn, sr):
    s = max(0, int(turn["start"] * sr))
    e = min(len(stream), int(turn["end"] * sr))
    return float(np.mean(stream[s:e].astype(np.float64) ** 2)) if e > s else 0.0

speaker_A, speaker_B = labels[0], labels[1]
turns_A = [t for t in speaker_turns if t["speaker"] == speaker_A]
turns_B = [t for t in speaker_turns if t["speaker"] == speaker_B]

def channel_purity_score(left_ch, right_ch, turns_A, turns_B, sr):
    e_A_L = np.mean([_energy_in_turn(left_ch,  t, sr) for t in turns_A])
    e_A_R = np.mean([_energy_in_turn(right_ch, t, sr) for t in turns_A])
    purity_A = e_A_L / (e_A_L + e_A_R + 1e-8)

    e_B_R = np.mean([_energy_in_turn(right_ch, t, sr) for t in turns_B])
    e_B_L = np.mean([_energy_in_turn(left_ch,  t, sr) for t in turns_B])
    purity_B = e_B_R / (e_B_R + e_B_L + 1e-8)
    return (purity_A + purity_B) / 2

purity = channel_purity_score(left_channel, right_channel, turns_A, turns_B, sample_rate)
print(f"📊 Channel Purity Score: {purity:.3f}")

📊 Channel Purity Score: 0.718


In [57]:
# Cell 6.2 — Quality gate + manifest
import json
from datetime import datetime, timezone

PURITY_THRESHOLD = 0.75
MANIFEST_PATH = "/content/manifest.jsonl"

if purity >= PURITY_THRESHOLD:
    entry = {
        "source_file": AUDIO_PATH,
        "output_file": "/content/dual_channel_output.wav",
        "duration_sec": round(len(left_channel) / sample_rate, 3),
        "sample_rate": OUTPUT_SR,                    # ← what the WAV file actually is
        "internal_sample_rate": sample_rate,         # ← what the pipeline ran at (16 kHz)
        "speaker_left": speaker_A,
        "speaker_right": speaker_B,
        "num_turns": len(speaker_turns),
        "purity_score": round(float(purity), 4),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(MANIFEST_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print("✅ Clip passed quality gate. Manifest updated.")
    print(json.dumps(entry, ensure_ascii=False, indent=2))
else:
    print(f"❌ Clip failed quality gate (purity {purity:.3f} < {PURITY_THRESHOLD}).")
    print("   Try: shorter clip, cleaner audio, or DB_CLEAR=3.0 in Cell 5.1.")

❌ Clip failed quality gate (purity 0.718 < 0.75).
   Try: shorter clip, cleaner audio, or DB_CLEAR=3.0 in Cell 5.1.


In [ ]:
# Cell 6.3 — Bundle outputs into a single ZIP (clean-room; output SR aware)
import os, shutil, zipfile, json
import numpy as np, soundfile as sf
import torch, torchaudio
from google.colab import files

INCLUDE_STEREO_MIX = True
INCLUDE_MANIFEST   = True
BUNDLE_DIR = "/content/bundle"

# --- 0. Resample to OUTPUT_SR (no-op if OUTPUT_SR == sample_rate) ---
if OUTPUT_SR != sample_rate:
    _rs = torchaudio.transforms.Resample(sample_rate, OUTPUT_SR)
    def _resample(x):
        t = torch.from_numpy(np.asarray(x, dtype=np.float32)).unsqueeze(0)
        return _rs(t).squeeze(0).numpy()
    out_left  = _resample(left_channel)
    out_right = _resample(right_channel)
    out_sr    = OUTPUT_SR
    print(f"↻ Resampled {sample_rate} → {OUTPUT_SR} Hz "
          f"(samples: {len(left_channel)} → {len(out_left)})")
else:
    out_left, out_right = left_channel, right_channel
    out_sr = sample_rate
    print(f"✓ No resample needed (output == internal SR == {out_sr} Hz)")

# --- 1. Clean-room bundle ---
if os.path.isdir(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)
os.makedirs(BUNDLE_DIR, exist_ok=True)

# --- 2. speaker_1.wav (Speaker A, mono) ---
sf.write(os.path.join(BUNDLE_DIR, "speaker_1.wav"),
         np.asarray(out_left, dtype=np.float32), out_sr, subtype="PCM_16")

# --- 3. speaker_2.wav (Speaker B, mono) ---
sf.write(os.path.join(BUNDLE_DIR, "speaker_2.wav"),
         np.asarray(out_right, dtype=np.float32), out_sr, subtype="PCM_16")

# --- 4. Original (current clip only) ---
shutil.copy2(AUDIO_PATH, os.path.join(BUNDLE_DIR, os.path.basename(AUDIO_PATH)))

# --- 5. Stereo mix ---
STEREO_PATH = "/content/dual_channel_output.wav"
if INCLUDE_STEREO_MIX:
    stereo = np.stack([out_left, out_right], axis=1)
    sf.write(STEREO_PATH, stereo, out_sr, subtype="PCM_16")
    shutil.copy2(STEREO_PATH, os.path.join(BUNDLE_DIR, "dual_channel_output.wav"))

# --- 6. Per-clip manifest entry only ---
if INCLUDE_MANIFEST and os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, encoding="utf-8") as f:
        entries = [json.loads(ln) for ln in f if ln.strip()]
    clip_entry = next(
        (e for e in reversed(entries) if e.get("source_file") == AUDIO_PATH), None,
    )
    if clip_entry is not None:
        with open(os.path.join(BUNDLE_DIR, "manifest.jsonl"), "w", encoding="utf-8") as f:
            f.write(json.dumps(clip_entry, ensure_ascii=False) + "\n")
        print(f"✅ Bundled manifest entry for: {AUDIO_PATH}")
    else:
        print(f"⚠️  No manifest entry matched {AUDIO_PATH} — bundle omits manifest.jsonl.")

# --- 7. Zip ---
ZIP_PATH = "/content/bangla_dual_channel_bundle.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(BUNDLE_DIR)):
        zf.write(os.path.join(BUNDLE_DIR, fname), arcname=fname)

print("\n📦 Bundle contents:")
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f"   • {info.filename:35s} {info.file_size/1024:8.1f} KB")
print(f"\n✅ {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1024/1024:.2f} MB)  @ {out_sr} Hz")
files.download(ZIP_PATH)
print("✅ Download triggered.")



### Reference Links
- Diarization: `pyannote/speaker-diarization-3.1` (loaded by `pyannote.audio==4.0.7`)
- Separation: https://github.com/meizhong986/ClearerVoice-Studio (fork)
- Original: https://github.com/modelscope/ClearerVoice-Studio